# Turkish Prompt Task Classification with BERT

This notebook trains a binary classifier to identify the task requested by a Turkish prompt:

- `0`: summarization
- `1`: named-entity extraction

The input contains the full prompt, including its instruction and text. The notebook samples 300 examples per class, creates reproducible train, validation, and test splits, fine-tunes a Turkish BERT model, and shows sample predictions.

The evaluation split uses the same prompt templates as training. Its score therefore measures performance on these prompt styles, not on unseen phrasings.

## Requirements

Use a Python environment with PyTorch installed for your hardware, plus `datasets` and `transformers`:

```bash
pip install datasets transformers
```

The notebook expects `data/processed/TR-News_12960_content_length_ratio_balanced.csv.gz` in the project, and downloads `turkish-nlp-suite/turkish-wikiNER` from the Hugging Face Hub. Run the notebook from the project folder or one of its subfolders.

In [ ]:
import csv
import gzip
import random
from collections import Counter, defaultdict
from pathlib import Path

SEED = 42
SAMPLES_PER_CLASS = 300

SOURCE_DATA_PATH = Path("data/processed/TR-News_12960_content_length_ratio_balanced.csv.gz")
CLASSIFIER_DATA_PATH = Path("data/processed/BERT_task_classifier_600.csv")
SPLIT_DIR = Path("data/processed/BERT_task_classifier_splits")

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / SOURCE_DATA_PATH).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not find {SOURCE_DATA_PATH}. Run this notebook from the project folder or a subfolder."
    )

SOURCE_DATA_PATH = project_root / SOURCE_DATA_PATH
CLASSIFIER_DATA_PATH = project_root / CLASSIFIER_DATA_PATH
SPLIT_DIR = project_root / SPLIT_DIR

print("Project root:", project_root)
print("Summary source:", SOURCE_DATA_PATH)
print("Classifier dataset:", CLASSIFIER_DATA_PATH)


## Prepare summarization examples

The source contains four summary-length styles. All are labeled `0` here because this classifier routes by task, not by summary length.

In [ ]:
SUMMARY_CLASSES = ["very_short", "short", "summary", "long"]
rng = random.Random(SEED)
summary_rows_by_class = defaultdict(list)

required_columns = {"prompt_class", "instruction", "content"}
with gzip.open(SOURCE_DATA_PATH, "rt", encoding="utf-8-sig", newline="") as source_file:
    reader = csv.DictReader(source_file)
    if reader.fieldnames is None:
        raise ValueError("The summary source has no header row.")

    missing_columns = required_columns.difference(reader.fieldnames)
    if missing_columns:
        raise ValueError(f"The summary source is missing columns: {sorted(missing_columns)}")

    for row in reader:
        prompt_class = row["prompt_class"]
        if prompt_class not in SUMMARY_CLASSES:
            raise ValueError(f"Unexpected summary class: {prompt_class}")
        if not row["instruction"].strip() or not row["content"].strip():
            continue
        summary_rows_by_class[prompt_class].append(row)

summary_examples = []
for prompt_class in SUMMARY_CLASSES:
    available_rows = summary_rows_by_class[prompt_class]
    if len(available_rows) < SAMPLES_PER_CLASS // len(SUMMARY_CLASSES):
        raise ValueError(f"Not enough examples for summary style: {prompt_class}")

    # Sample equally across summary styles to preserve instruction variety.
    per_style_count = SAMPLES_PER_CLASS // len(SUMMARY_CLASSES)
    for row in rng.sample(available_rows, per_style_count):
        full_prompt = f"{row['instruction'].strip()}\n\n{row['content'].strip()}"
        summary_examples.append({"text": full_prompt, "label": 0})

print(f"Prepared {len(summary_examples)} summarization examples.")


## Prepare named-entity extraction examples

The examples come from the WikiNER training split. Entity-focused instructions are assigned label `1`; generic keyword and information-extraction instructions are excluded because they do not consistently mean named-entity extraction.

In [ ]:
from datasets import load_dataset

ENTITY_INSTRUCTIONS = [
    "Metindeki varlıkları bul.",
    "Varlıkları türleriyle listele.",
    "Tanımlı etiketlerdeki varlıkları çıkar.",
    "Kişi, yer, tarih, kurum, grup, unvan ve olay adlarını tespit et.",
    "Cümlede geçen hedef varlıkları belirle.",
    "Varlıkları belirtilen kategorilere göre sınıflandır.",
]
TARGET_ENTITY_TYPES = {"PERSON", "GPE", "DATE", "ORG", "NORP", "TITLE", "EVENT"}

raw_dataset = load_dataset("turkish-nlp-suite/turkish-wikiNER")
train_split = raw_dataset["train"]

tag_feature = train_split.features["tags"]
while hasattr(tag_feature, "feature"):
    tag_feature = tag_feature.feature

raw_label_names = getattr(tag_feature, "names", None)
if raw_label_names is None:
    raise ValueError("WikiNER tag names are unavailable; cannot safely decode entity labels.")


def decode_tag(raw_tag):
    if isinstance(raw_tag, str) and not raw_tag.strip().lstrip("-").isdigit():
        return raw_tag.upper()

    tag_id = int(raw_tag)
    if not 0 <= tag_id < len(raw_label_names):
        raise ValueError(f"Tag ID {tag_id} is outside the WikiNER label map.")
    return str(raw_label_names[tag_id]).upper()


def target_types_in_sentence(tags):
    found_types = set()
    for raw_tag in tags:
        tag = decode_tag(raw_tag)
        if "-" not in tag:
            continue
        prefix, entity_type = tag.split("-", 1)
        if prefix in {"B", "I"} and entity_type in TARGET_ENTITY_TYPES:
            found_types.add(entity_type)
    return found_types


def detokenize(tokens):
    attach_before = {".", ",", "!", "?", ";", ":", ")", "]", "}", "%", "'", "’"}
    attach_after = {"(", "[", "{", "'", "‘"}
    text = ""
    previous_token = ""

    for token in tokens:
        if not text:
            text = token
        elif token in attach_before or previous_token in attach_after:
            text += token
        else:
            text += " " + token
        previous_token = token

    return text


# Reduce the three frequent PERSON/GPE combinations, then retain a small number of empty examples.
rebalanced_categories = {
    frozenset({"PERSON"}),
    frozenset({"GPE"}),
    frozenset({"PERSON", "GPE"}),
}
category_indices = {category: [] for category in rebalanced_categories}
positive_rows = []
empty_rows = []

for row_index, row in enumerate(train_split):
    entity_types = target_types_in_sentence(row["tags"])
    if entity_types:
        positive_rows.append((row_index, row))
        category = frozenset(entity_types)
        if category in category_indices:
            category_indices[category].append(row_index)
    else:
        empty_rows.append(row)

indices_to_remove = set()
category_rng = random.Random(SEED)
for indices in category_indices.values():
    removal_count = round(len(indices) * 0.68)
    indices_to_remove.update(category_rng.sample(indices, removal_count))

retained_positive_rows = [
    row for row_index, row in positive_rows if row_index not in indices_to_remove
]
empty_keep_count = round(len(retained_positive_rows) * 0.05 / 0.95)
retained_empty_rows = random.Random(SEED).sample(
    empty_rows,
    min(empty_keep_count, len(empty_rows)),
)
entity_candidate_rows = retained_positive_rows + retained_empty_rows

if len(entity_candidate_rows) < SAMPLES_PER_CLASS:
    raise ValueError(
        f"Only {len(entity_candidate_rows)} entity examples are available; "
        f"{SAMPLES_PER_CLASS} are required."
    )

sampled_entity_rows = random.Random(SEED).sample(entity_candidate_rows, SAMPLES_PER_CLASS)
instruction_pool = [
    instruction
    for instruction in ENTITY_INSTRUCTIONS
    for _ in range(SAMPLES_PER_CLASS // len(ENTITY_INSTRUCTIONS))
]
if len(instruction_pool) != SAMPLES_PER_CLASS:
    raise ValueError("The entity instruction count must divide the requested sample size.")
random.Random(SEED).shuffle(instruction_pool)

entity_examples = []
for row, instruction in zip(sampled_entity_rows, instruction_pool):
    sentence = detokenize(row["tokens"])
    full_prompt = f"{instruction}\n\nCümle: {sentence}"
    entity_examples.append({"text": full_prompt, "label": 1})

print(f"Prepared {len(entity_examples)} named-entity extraction examples.")


## Combine and split the data

The combined dataset contains only the full prompt and its binary label. The split is stratified by label and reproducible with the seed above.

In [ ]:
all_examples = summary_examples + entity_examples
rng.shuffle(all_examples)

label_counts = Counter(example["label"] for example in all_examples)
if label_counts != {0: SAMPLES_PER_CLASS, 1: SAMPLES_PER_CLASS}:
    raise ValueError(f"Unexpected class counts: {dict(label_counts)}")

CLASSIFIER_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
with CLASSIFIER_DATA_PATH.open("w", encoding="utf-8-sig", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=["text", "label"])
    writer.writeheader()
    writer.writerows(all_examples)

examples_by_label = defaultdict(list)
for example in all_examples:
    examples_by_label[example["label"]].append(example)

splits = {"train": [], "validation": [], "test": []}
for label, examples in examples_by_label.items():
    examples = list(examples)
    rng.shuffle(examples)
    train_end = int(len(examples) * 0.8)
    validation_end = train_end + int(len(examples) * 0.1)
    splits["train"].extend(examples[:train_end])
    splits["validation"].extend(examples[train_end:validation_end])
    splits["test"].extend(examples[validation_end:])

for split_rows in splits.values():
    rng.shuffle(split_rows)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for split_name, split_rows in splits.items():
    split_path = SPLIT_DIR / f"{split_name}.csv"
    with split_path.open("w", encoding="utf-8-sig", newline="") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=["text", "label"])
        writer.writeheader()
        writer.writerows(split_rows)

    counts = Counter(row["label"] for row in split_rows)
    print(f"{split_name}: {len(split_rows)} rows; labels={dict(counts)}")

print("Combined dataset:", CLASSIFIER_DATA_PATH)
print("Split directory:", SPLIT_DIR)


## Fine-tune the classifier

The classification head predicts whether the full prompt requests summarization or entity extraction. Inputs are truncated to 512 tokens when needed; the instruction is at the start of each prompt and is retained.

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

MODEL_NAME = "ytu-ce-cosmos/turkish-small-bert-uncased"
OUTPUT_DIR = project_root / "outputs/turkish_small_bert_task_classifier"
MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
LABEL_NAMES = {0: "summarization", 1: "entity_extraction"}

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# The uncased Turkish model requires this Turkish-aware lowercase conversion.
def normalize_turkish(text):
    return text.replace("I", "ı").lower()


def read_classifier_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as input_file:
        rows = list(csv.DictReader(input_file))
    if not rows:
        raise ValueError(f"The split file is empty: {path}")
    return rows


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)


class PromptDataset(Dataset):
    def __init__(self, path):
        self.examples = []
        self.truncated_count = 0
        for row in read_classifier_rows(path):
            text = normalize_turkish(row["text"])
            token_count = len(tokenizer.tokenize(text)) + tokenizer.num_special_tokens_to_add(pair=False)
            if token_count > MAX_LENGTH:
                self.truncated_count += 1

            encoded = tokenizer(
                text,
                truncation=True,
                max_length=MAX_LENGTH,
                padding=False,
            )
            self.examples.append({
                "input_ids": encoded["input_ids"],
                "attention_mask": encoded["attention_mask"],
                "label": int(row["label"]),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(items):
    inputs = [
        {"input_ids": item["input_ids"], "attention_mask": item["attention_mask"]}
        for item in items
    ]
    batch = tokenizer.pad(inputs, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor([item["label"] for item in items], dtype=torch.long)
    return batch


train_dataset = PromptDataset(SPLIT_DIR / "train.csv")
validation_dataset = PromptDataset(SPLIT_DIR / "validation.csv")
test_dataset = PromptDataset(SPLIT_DIR / "test.csv")

for split_name, dataset in [
    ("train", train_dataset),
    ("validation", validation_dataset),
    ("test", test_dataset),
]:
    print(f"{split_name}: {len(dataset)} rows; truncated={dataset.truncated_count}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=LABEL_NAMES,
    label2id={name: index for index, name in LABEL_NAMES.items()},
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)


def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    correct = 0
    counts = {label: {"tp": 0, "fp": 0, "fn": 0} for label in LABEL_NAMES}

    with torch.no_grad():
        for batch in dataloader:
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)
            labels = batch["labels"]
            predictions = outputs.logits.argmax(dim=-1)
            batch_size = labels.size(0)
            total_loss += outputs.loss.item() * batch_size
            total_examples += batch_size
            correct += (predictions == labels).sum().item()

            for true_label, predicted_label in zip(labels.tolist(), predictions.tolist()):
                if true_label == predicted_label:
                    counts[true_label]["tp"] += 1
                else:
                    counts[predicted_label]["fp"] += 1
                    counts[true_label]["fn"] += 1

    f1_scores = []
    for label in LABEL_NAMES:
        tp = counts[label]["tp"]
        fp = counts[label]["fp"]
        fn = counts[label]["fn"]
        denominator = 2 * tp + fp + fn
        f1_scores.append(2 * tp / denominator if denominator else 0.0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": correct / total_examples,
        "macro_f1": sum(f1_scores) / len(f1_scores),
    }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_validation_f1 = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    total_train_examples = 0

    for batch in train_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        batch_size = batch["labels"].size(0)
        total_train_loss += outputs.loss.item() * batch_size
        total_train_examples += batch_size

    validation_metrics = evaluate(model, validation_loader)
    train_loss = total_train_loss / total_train_examples
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} | "
        f"val_loss={validation_metrics['loss']:.4f} | "
        f"val_accuracy={validation_metrics['accuracy']:.4f} | "
        f"val_macro_f1={validation_metrics['macro_f1']:.4f}"
    )

    if validation_metrics["macro_f1"] > best_validation_f1:
        best_validation_f1 = validation_metrics["macro_f1"]
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print("Best model saved.")

best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
best_model.to(device)
test_metrics = evaluate(best_model, test_loader)

print("\nTest results:")
for metric_name, value in test_metrics.items():
    print(f"{metric_name}: {value:.4f}")
print("Model directory:", OUTPUT_DIR)

with (OUTPUT_DIR / "test_metrics.json").open("w", encoding="utf-8") as metrics_file:
    json.dump(test_metrics, metrics_file, ensure_ascii=False, indent=2)


## Inspect sample predictions

The following cell selects two test examples per class and compares the model prediction with the expected label.

In [ ]:
import random
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = project_root / "outputs/turkish_small_bert_task_classifier"
TEST_FILE = SPLIT_DIR / "test.csv"
LABEL_NAMES = {0: "summarization", 1: "entity_extraction"}

# Load the fine-tuned model and its tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, do_lower_case=False)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

test_rows = read_classifier_rows(TEST_FILE)
rows_by_label = {0: [], 1: []}
for row in test_rows:
    rows_by_label[int(row["label"])].append(row)

sample_rng = random.Random(SEED)
selected_rows = []
for label in LABEL_NAMES:
    selected_rows.extend(sample_rng.sample(rows_by_label[label], min(2, len(rows_by_label[label]))))
sample_rng.shuffle(selected_rows)

for index, row in enumerate(selected_rows, start=1):
    text = row["text"]
    expected_label = int(row["label"])
    inputs = tokenizer(
        normalize_turkish(text),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0]

    predicted_label = int(probabilities.argmax().item())
    confidence = float(probabilities[predicted_label].item())
    preview = text.replace("\n", " ")
    if len(preview) > 500:
        preview = preview[:500] + "..."

    print(f"\n--- Example {index} ---")
    print(f"Expected: {LABEL_NAMES[expected_label]}")
    print(f"Predicted: {LABEL_NAMES[predicted_label]}")
    print(f"Confidence: {confidence:.2%}")
    print("Prompt preview:")
    print(preview)
